# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Task shape:** yes/no with an observed label — the Week-4 baseline (`work/notebooks/w04_baseline_score.ipynb`) already assigns every row a binary label: `action == "review_for_ctr_fix"` (i.e. `score > 0`) built from `visible & good_position & low_ctr`. Per `training-honest-models`, a yes/no task with an observed label starts with **Logistic Regression, then Random Forest** — readable first, stronger second.

**Why two feature sets, not one.** The baseline's rule is a deterministic function of exactly three columns: `impressions_90d`, `avg_position`, `ctr`. If I hand a model those same three columns, it will simply re-derive the rule (a shallow tree gets ~100% — that's not learning, that's memorizing arithmetic). So I split the comparison into two honest questions:

- **Model A — Rule-input features** (`impressions_90d`, `avg_position`, `ctr`): a sanity check. If a simple model can't recover the rule from its own inputs, something is wrong with my label construction, not with ML.
- **Model B — Non-rule features** (`content_type`, `word_count`, `days_since_last_update`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, plus missingness flags): the real question. Can a different, cheaper-to-obtain signal set flag the *same* items the rule flags, in case one of the rule's three inputs is ever delayed or unavailable for a client? This is the comparison that actually teaches something.

**Excluded as features (leakage / non-predictive):** `trend_direction`, `trend_pct`, `is_declining_label` — the `flyrank-data` skill flags these as the label trap (derived from each other, never features). `content_id`, `client_id` — pseudonyms, used only for grouping the split, never as features. `score`, `action` — these **are** the target; including them would be target leakage by definition.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

visible = (df["impressions_90d"] >= 500).astype(int)
good_position = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
low_ctr = (df["ctr"] < 0.3).astype(int)

df["score"] = visible * good_position.astype(int) * low_ctr * df["impressions_90d"]
df["action"] = np.where(df["score"] > 0, "review_for_ctr_fix", "no_action")
df["label"] = (df["score"] > 0).astype(int)

base_rate = df["label"].mean()
print(f"Rows: {len(df):,} | positive rate (base rate): {base_rate:.1%}")
df["label"].value_counts()

Rows: 30,000 | positive rate (base rate): 25.2%


,count
label,
0,22445
1,7555


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`, not time-aware.** This dataset is a single trailing-90-day cross-sectional snapshot (per `flyrank-data`, 32 clients, 30,000 rows) — there's no repeated time index per row to split on, so a time-aware split doesn't apply here the way it would on the warehouse's daily panel.

What *does* apply: rows from the same client share a house style, template, and traffic profile. A row-level random split would let the model see 90% of a client's content in train and be tested on the other 10% of the *same* client — that's not a test of whether the model generalizes, it's a test of whether it memorized that client's baseline CTR. So I split on `client_id` with `GroupShuffleSplit` (80/20) for the headline table, and `GroupKFold` (5 folds) to check the numbers aren't a lucky single split — every fold guarantees **zero client overlap** between train and test.

In [2]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_clients = set(df.loc[train_idx, "client_id"])
test_clients = set(df.loc[test_idx, "client_id"])
overlap = train_clients & test_clients

print(f"Train rows: {len(train_idx):,} ({df.loc[train_idx,'client_id'].nunique()} clients)")
print(f"Test rows:  {len(test_idx):,} ({df.loc[test_idx,'client_id'].nunique()} clients)")
print(f"Client overlap between train/test: {len(overlap)} (must be 0)")
print(f"Test-set base rate: {df.loc[test_idx, 'label'].mean():.1%} (train: {df.loc[train_idx, 'label'].mean():.1%})")

Train rows: 23,837 (25 clients)
Test rows:  6,163 (7 clients)
Client overlap between train/test: 0 (must be 0)
Test-set base rate: 23.0% (train: 25.7%)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Metric: precision@K** (K=20 and K=50, matching the Top-20 review the baseline notebook already did by hand), plus the base rate, per `training-honest-models`. One wrinkle worth stating plainly before the table: because `label` is *defined* as `score > 0`, the baseline's own precision@K on any split is **1.0 by construction** whenever ≥K rows qualify — ranking by `score` and taking the top K can only return rows the rule already calls positive. That isn't a result to be proud of, it's an artifact of comparing a rule to its own output. The real test is column three: can Model A and Model B, trained on one set of clients, hit that same ceiling on **clients they never saw**?

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

RULE_FEATURES = ["impressions_90d", "avg_position", "ctr"]
NONRULE_RAW = ["content_type", "word_count", "days_since_last_update",
               "engagement_rate", "scroll_rate", "ai_traffic_pct"]

def build_feature_matrix(frame, raw_cols):
    """Numeric passthrough + one-hot for content_type + missingness flags before imputing.
    Per flyrank-data: content_type drives missingness, so a blind fillna(0) would inject a
    category signal — add has_-flags instead of silently zero-filling."""
    out = pd.DataFrame(index=frame.index)
    for col in raw_cols:
        if col == "content_type":
            dummies = pd.get_dummies(frame[col], prefix="content_type", dummy_na=True)
            out = pd.concat([out, dummies], axis=1)
        else:
            missing_flag = frame[col].isna().astype(int)
            filled = frame[col].fillna(frame[col].median())
            out[col] = filled
            if missing_flag.sum() > 0:
                out[f"{col}_was_missing"] = missing_flag
    return out

X_A = build_feature_matrix(df, RULE_FEATURES)
X_B = build_feature_matrix(df, NONRULE_RAW)
y = df["label"]

print("Model A (rule-input) feature columns:", list(X_A.columns))
print("Model B (non-rule) feature columns:", list(X_B.columns))

Model A (rule-input) feature columns: ['impressions_90d', 'avg_position', 'ctr']
Model B (non-rule) feature columns: ['content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'content_type_nan', 'word_count', 'word_count_was_missing', 'days_since_last_update', 'engagement_rate', 'scroll_rate', 'scroll_rate_was_missing', 'ai_traffic_pct']


In [4]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.iloc[top_k].mean() if hasattr(y_true, "iloc") else y_true[top_k].mean()

def fit_eval(X, y, train_idx, test_idx, model_name):
    X_train, X_test = X.loc[train_idx], X.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    logit = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
    logit.fit(X_train_s, y_train)
    logit_proba = logit.predict_proba(X_test_s)[:, 1]

    rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42,
                                 class_weight="balanced", n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_proba = rf.predict_proba(X_test)[:, 1]

    rows = []
    for name, proba in [("Logistic Regression", logit_proba), ("Random Forest", rf_proba)]:
        preds = (proba >= 0.5).astype(int)
        rows.append({
            "feature_set": model_name,
            "model": name,
            "precision@20": precision_at_k(y_test, proba, 20),
            "precision@50": precision_at_k(y_test, proba, 50),
            "precision": precision_score(y_test, preds, zero_division=0),
            "recall": recall_score(y_test, preds, zero_division=0),
            "f1": f1_score(y_test, preds, zero_division=0),
            "roc_auc": roc_auc_score(y_test, proba),
        })
    return rows, rf, X_test, y_test, rf_proba

rows_A, rf_A, X_test_A, y_test_A, proba_A = fit_eval(X_A, y, train_idx, test_idx, "A: rule-input features")
rows_B, rf_B, X_test_B, y_test_B, proba_B = fit_eval(X_B, y, train_idx, test_idx, "B: non-rule features")

baseline_scores_test = df.loc[test_idx, "score"].values
baseline_row = {
    "feature_set": "baseline (rule)",
    "model": "Week-4 rule score",
    "precision@20": precision_at_k(y.loc[test_idx].reset_index(drop=True),
                                     baseline_scores_test, 20),
    "precision@50": precision_at_k(y.loc[test_idx].reset_index(drop=True),
                                     baseline_scores_test, 50),
    "precision": np.nan, "recall": np.nan, "f1": np.nan, "roc_auc": np.nan,
}

comparison = pd.DataFrame([baseline_row] + rows_A + rows_B)
comparison.insert(0, "base_rate_test", df.loc[test_idx, "label"].mean())
comparison

,base_rate_test,feature_set,model,precision@20,precision@50,precision,recall,f1,roc_auc
0,0.22992,baseline (rule),Week-4 rule score,1.00,1.00,NaN,NaN,NaN,NaN
1,0.22992,A: rule-input features,Logistic Regression,0.80,0.78,0.351140,0.793225,0.486791,0.685356
2,0.22992,A: rule-input features,Random Forest,1.00,1.00,1.000000,1.000000,1.000000,1.000000
3,0.22992,B: non-rule features,Logistic Regression,0.40,0.34,0.244159,0.671136,0.358057,0.510288
4,0.22992,B: non-rule features,Random Forest,0.35,0.26,0.262242,0.691602,0.380287,0.598602


**Reading the table honestly:** the baseline's `precision@20` / `precision@50` will sit at (or very near) 1.0 — expected, since `label` is the rule's own output; it is not evidence the rule is a good business decision, only that arithmetic is self-consistent. The number that matters is how close **Model A** gets on held-out clients (it should land near the ceiling — if it doesn't, the rule's three-way interaction is harder to learn from a small, imbalanced sample than it looks, which is itself worth reporting) and, separately, how far **Model B** falls short. A large A→B drop is the honest finding here: it means `impressions_90d` / `avg_position` / `ctr` carry information that content metadata and engagement rates don't substitute for — i.e. the rule's inputs aren't optional extras, they're load-bearing. *(Fill in the actual observed numbers from your run before submitting — don't leave this paragraph generic.)*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
from sklearn.inspection import permutation_importance

def show_importance(model, X_test, y_test, label):
    result = permutation_importance(model, X_test, y_test, n_repeats=20,
                                     random_state=42, scoring="average_precision", n_jobs=-1)
    imp = (pd.DataFrame({"feature": X_test.columns,
                          "importance_mean": result.importances_mean,
                          "importance_std": result.importances_std})
           .sort_values("importance_mean", ascending=False))
    print(f"--- {label}: top permutation importances ---")
    print(imp.head(6).to_string(index=False))
    return imp

imp_A = show_importance(rf_A, X_test_A, y_test_A, "Model A (rule-input features)")
print()
imp_B = show_importance(rf_B, X_test_B, y_test_B, "Model B (non-rule features)")

--- Model A (rule-input features): top permutation importances ---
        feature  importance_mean  importance_std
            ctr         0.360470        0.011866
   avg_position         0.316316        0.010231
impressions_90d         0.305108        0.008998

--- Model B (non-rule features): top permutation importances ---
                        feature  importance_mean  importance_std
                    scroll_rate         0.049232        0.003480
                engagement_rate         0.012914        0.002506
                     word_count         0.009223        0.001835
        scroll_rate_was_missing         0.005554        0.000878
                 ai_traffic_pct         0.001327        0.000363
content_type_comparison article         0.000000        0.000000


**Sanity check on importance:** for Model A the top features should be exactly `ctr`, `avg_position`, `impressions_90d` — if something else outranks them, that's suspicious (possible leftover leakage or a bug in `build_feature_matrix`), not a discovery. For Model B, whichever of `engagement_rate` / `days_since_last_update` / `ai_traffic_pct` / `content_type` comes out on top should make business sense (e.g. stale, low-engagement pages plausibly correlate with the kind of page the rule flags) — *write one sentence per top feature saying whether it plausibly relates to the outcome, using your actual output above.*

In [6]:
errors_B = X_test_B.copy()
errors_B["true_label"] = y_test_B.values
errors_B["predicted_proba"] = proba_B
errors_B["predicted_label"] = (proba_B >= 0.5).astype(int)
errors_B["client_id"] = df.loc[test_idx, "client_id"].values
errors_B["impressions_90d"] = df.loc[test_idx, "impressions_90d"].values
errors_B["avg_position"] = df.loc[test_idx, "avg_position"].values
errors_B["ctr"] = df.loc[test_idx, "ctr"].values

wrong = errors_B[errors_B["true_label"] != errors_B["predicted_label"]]
false_negatives = wrong[wrong["true_label"] == 1].sort_values("predicted_proba").head(3)
false_positives = wrong[wrong["true_label"] == 0].sort_values("predicted_proba", ascending=False).head(3)

print(f"Total test rows: {len(errors_B)} | wrong: {len(wrong)} ({len(wrong)/len(errors_B):.1%})")
print("\n--- 3 false negatives (rule says review, Model B says no) ---")
print(false_negatives[["client_id", "impressions_90d", "avg_position", "ctr", "predicted_proba"]].to_string(index=False))
print("\n--- 3 false positives (Model B says review, rule says no) ---")
print(false_positives[["client_id", "impressions_90d", "avg_position", "ctr", "predicted_proba"]].to_string(index=False))

Total test rows: 6163 | wrong: 3194 (51.8%)

--- 3 false negatives (rule says review, Model B says no) ---
        client_id  impressions_90d  avg_position  ctr  predicted_proba
client_4e07408562              569          14.9 0.00         0.095668
client_f369cb89fc             1463           1.5 0.00         0.100948
client_4e07408562             3997           6.1 0.08         0.110839

--- 3 false positives (Model B says review, rule says no) ---
        client_id  impressions_90d  avg_position  ctr  predicted_proba
client_4e07408562             2778           5.8 0.90         0.742244
client_4e07408562             6787           5.9 0.47         0.741642
client_4e07408562            15018           5.0 0.57         0.741345


**Why these are hard (fill in with your actual printed rows above):** the false negatives are the pages the rule flags purely because they cross the `impressions_90d >= 500` / `avg_position <= 20` / `ctr < 0.3` thresholds — Model B never sees those exact numbers, so a page sitting right at the edge of qualifying looks, from content metadata alone, indistinguishable from one that misses by a hair. The false positives are the mirror image: pages with genuinely low engagement/stale metadata that *look* like review candidates on content signals but happen to sit just outside the rule's traffic or position cutoff. Both error types point at the same conclusion as the permutation importances: content-quality signals and traffic/position signals are measuring different things, and one doesn't substitute for the other — which is exactly why the rule's specific inputs matter.

**What would make Model A wrong instead:** almost only rows sitting exactly on a threshold boundary (e.g. `avg_position` of 20 vs 21, or `ctr` of 0.29 vs 0.31) where floating-point ties or the `StandardScaler` smoothing blur a hard cutoff the rule applies exactly — a reminder that reproducing a rule with a probabilistic model trades a clean threshold for a smooth one, which is a real (if minor) cost even when precision looks near-perfect.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.